# 17b_generate_mel_nv_cue_qualitative_and_cue_metrics

Cued MEL/NV qualitative evaluation and cue localization metrics.

This notebook:

1. Creates a small qualitative CSV from the **cued** MEL/NV test split.
2. Generates qualitative panels only for cue-trained models:
   - Cue CE
   - Cue HA
3. Computes cue localization metrics on MEL test images using `cue_mask_rel_path`:
   - cue energy fraction
   - cue pointing game
   - lesion energy fraction

Run this notebook from the repo `notebooks/` folder.


In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd
import numpy as np
from PIL import Image
import cv2

QUAL_SEED = 42
CUE_SIZE = "small"      # small, big
CUE_PLACEMENT = "fixed"  # fixed, random
TARGET_BLOCK_INDEX = -4

REPO_ROOT = Path("..").resolve()
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"

if CUE_PLACEMENT == "fixed":
    EXPERIMENT_NAME = f"mel_nv_fixed_center_seed{QUAL_SEED}"
    CUE_CSV_NAME = f"ham_mel_nv_cue_on_mel_fixed_center_seed{QUAL_SEED}.csv"
    CUE_QUAL_NAME = f"ham_mel_nv_cue_fixed_qualitative_10_seed{QUAL_SEED}.csv"
elif CUE_PLACEMENT == "random":
    EXPERIMENT_NAME = f"mel_nv_random_location_seed{QUAL_SEED}"
    CUE_CSV_NAME = f"ham_mel_nv_cue_on_mel_random_location_seed{QUAL_SEED}.csv"
    CUE_QUAL_NAME = f"ham_mel_nv_cue_random_qualitative_10_seed{QUAL_SEED}.csv"
else:
    raise ValueError("CUE_PLACEMENT must be 'fixed' or 'random'.")

SYN_ROOT = HAM_ROOT / f"synthetic_cue_{CUE_SIZE}" / EXPERIMENT_NAME

CUE_CSV = SYN_ROOT / "csv" / CUE_CSV_NAME
CUE_QUAL_CSV = SYN_ROOT / "csv" / CUE_QUAL_NAME

QUAL_ROOT = REPO_ROOT / "outputs" / f"qual_cue_mel_nv_{CUE_SIZE}_{CUE_PLACEMENT}_seed{QUAL_SEED}_block{TARGET_BLOCK_INDEX}"
QUAL_ROOT.mkdir(parents=True, exist_ok=True)

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

print("REPO_ROOT:", REPO_ROOT)
print("CUE_CSV:", CUE_CSV)
print("CUE_QUAL_CSV:", CUE_QUAL_CSV)
print("QUAL_ROOT:", QUAL_ROOT)

REPO_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis
CUE_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_on_mel_fixed_center_seed42.csv
CUE_QUAL_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_fixed_qualitative_10_seed42.csv
QUAL_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_cue_mel_nv_small_fixed_seed42_block-4


## 1. Create cued qualitative CSV

Select 5 MEL and 5 NV test images from the cued CSV. MEL images contain the green cue. NV images do not.


In [2]:
df = pd.read_csv(CUE_CSV)

required_cols = ["image_id", "gt_label", "split", "image_rel_path", "mask_rel_path", "cue_applied", "cue_mask_rel_path"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in {CUE_CSV}: {missing}")

test_df = df[df["split"] == "test"].copy()
mel = test_df[test_df["gt_label"] == "MEL"].sample(n=5, random_state=QUAL_SEED)
nv = test_df[test_df["gt_label"] == "NV"].sample(n=5, random_state=QUAL_SEED)

qual_df = pd.concat([mel, nv], axis=0).copy()
qual_df["order"] = qual_df["gt_label"].map({"MEL": 0, "NV": 1})
qual_df = qual_df.sort_values(["order", "image_id"]).drop(columns=["order"])

CUE_QUAL_CSV.parent.mkdir(parents=True, exist_ok=True)
qual_df.to_csv(CUE_QUAL_CSV, index=False)

print("Saved:", CUE_QUAL_CSV)
print("Rows:", len(qual_df))
print("Counts:")
print(qual_df.groupby(["gt_label", "cue_applied"]).size())

display(qual_df[["image_id", "gt_label", "split", "image_rel_path", "mask_rel_path", "cue_applied", "cue_mask_rel_path"]])

Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_fixed_qualitative_10_seed42.csv
Rows: 10
Counts:
gt_label  cue_applied
MEL       True           5
NV        False          5
dtype: int64


,image_id,gt_label,split,image_rel_path,mask_rel_path,cue_applied,cue_mask_rel_path
0,ISIC_0024459,MEL,test,synthetic_cue_small/mel_nv_fixed_center_seed42...,../HAM10000_segmentations_lesion_tschandl/ISIC...,True,synthetic_cue_small/mel_nv_fixed_center_seed42...
4,ISIC_0024756,MEL,test,synthetic_cue_small/mel_nv_fixed_center_seed42...,../HAM10000_segmentations_lesion_tschandl/ISIC...,True,synthetic_cue_small/mel_nv_fixed_center_seed42...
22,ISIC_0026993,MEL,test,synthetic_cue_small/mel_nv_fixed_center_seed42...,../HAM10000_segmentations_lesion_tschandl/ISIC...,True,synthetic_cue_small/mel_nv_fixed_center_seed42...
49,ISIC_0030798,MEL,test,synthetic_cue_small/mel_nv_fixed_center_seed42...,../HAM10000_segmentations_lesion_tschandl/ISIC...,True,synthetic_cue_small/mel_nv_fixed_center_seed42...
54,ISIC_0031408,MEL,test,synthetic_cue_small/mel_nv_fixed_center_seed42...,../HAM10000_segmentations_lesion_tschandl/ISIC...,True,synthetic_cue_small/mel_nv_fixed_center_seed42...
70,ISIC_0024416,NV,test,images/ISIC_0024416.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,False,NaN
74,ISIC_0024874,NV,test,images/ISIC_0024874.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,False,NaN
92,ISIC_0027312,NV,test,images/ISIC_0027312.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,False,NaN
119,ISIC_0029892,NV,test,images/ISIC_0029892.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,False,NaN
124,ISIC_0030763,NV,test,images/ISIC_0030763.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...,False,NaN


## 2. Define cue-trained experiments

In [3]:
EXPERIMENTS_CUE = {
    "Cue CE": {
        "checkpoint": REPO_ROOT / "external" / f"checkpoints2_{CUE_SIZE}" / "checkpoint-best-cue.pth",
        "checkpoint_model_type": "panderm",
        "use_seg_gate": False,
        "out_dir": QUAL_ROOT / "cam_cue_ce_on_cued_test",
    },
    "Cue HA": {
        "checkpoint": REPO_ROOT / "external" / f"checkpoints2_{CUE_SIZE}" / "checkpoint-best-cue-ha.pth",
        "checkpoint_model_type": "panderm",
        "use_seg_gate": False,
        "out_dir": QUAL_ROOT / "cam_cue_ha_on_cued_test",
    },
}

for name, cfg in EXPERIMENTS_CUE.items():
    print(name, "->", cfg["checkpoint"])
    if not cfg["checkpoint"].exists():
        print("  [WARN] missing checkpoint. Adjust path above.")

Cue CE -> /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-cue.pth
Cue HA -> /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-cue-ha.pth


## 3. Helper functions

In [4]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def generate_qualitative_cams(experiments: dict, csv_path: Path, img_dir: Path, num_samples: int, dry_run: bool = False):
    for exp_name, cfg in experiments.items():
        out_dir = cfg["out_dir"]
        out_dir.mkdir(parents=True, exist_ok=True)

        panel_items = "rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam"

        cmd = [
            "python", "-m", "scripts.generate_finer_cam_panderm",
            "--csv", str(csv_path),
            "--image_col", "image_rel_path",
            "--img_dir", str(img_dir),
            "--gt_col", "gt_label",
            "--checkpoint", str(cfg["checkpoint"]),
            "--checkpoint_model_type", cfg["checkpoint_model_type"],
            "--class_names", "MEL,NV",
            "--out_dir", str(out_dir),
            "--num_samples", str(num_samples),
            "--method", "finercam",
            "--compare_mode", "gt_pair",
            "--A", "MEL",
            "--B", "NV",
            "--topk_compare", "1",
            "--alpha", "0.8",
            "--panel_items", panel_items,
            "--mask_root", str(MASK_ROOT),
            "--mask_col", "mask_rel_path",
            # "--save_json",
            "--save_raw_cams",
            "--target_block_index", str(TARGET_BLOCK_INDEX),
        ]

        print(f"\nRunning cue CAM generation: {exp_name}")
        run_command(cmd, dry_run=dry_run)


def build_qualitative_pdf(csv_path: Path, experiments_json_path: Path, out_pdf: Path, num_samples: int = 10, dry_run: bool = False):
    out_pdf.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "-m", "scripts.make_qualitative_comparison_pdf",
        "--csv", str(csv_path),
        "--image_col", "image_rel_path",
        "--gt_col", "gt_label",
        "--out_pdf", str(out_pdf),
        "--experiments_json_path", str(experiments_json_path),
        "--num_samples", str(num_samples),
        "--missing_policy", "placeholder",
    ]
    run_command(cmd, dry_run=dry_run)


def resolve_rel_path(root: Path, rel_or_abs: str) -> Path:
    p = Path(str(rel_or_abs))
    if p.is_absolute():
        return p
    return (root / p).resolve()


def load_mask_as_bool(path: Path) -> np.ndarray:
    arr = np.array(Image.open(path).convert("L"))
    return arr > 0


# def load_cam_array(cam_dir: Path, image_id: str, candidates=("finercam", "map_diff", "gradcam_a")):
#     """Try common CAM npy filenames. Adjust here if your script saves different names."""
#     for stem in candidates:
#         patterns = [
#             f"{image_id}*{stem}*.npy",
#             f"*{image_id}*{stem}*.npy",
#             f"{image_id}_{stem}.npy",
#         ]
#         for pat in patterns:
#             matches = sorted(cam_dir.glob(pat))
#             if matches:
#                 return np.load(matches[0]), matches[0]
#     return None, None

def load_cam_array(cam_dir: Path, image_id: str, candidates=("finercam", "map_diff", "gradcam_a")):
    """Load raw CAM arrays saved by scripts.generate_finer_cam_panderm.

    Supports:
      1. cam_dir/raw_cams/ISIC_xxx/finercam.npy
      2. cam_dir/raw_cams/ISIC_xxx_finercam.npy
      3. fallback direct search under cam_dir
    """
    raw_dir = cam_dir / "raw_cams"

    for stem in candidates:
        exact_candidates = [
            raw_dir / image_id / f"{stem}.npy",
            raw_dir / f"{image_id}_{stem}.npy",
            cam_dir / image_id / f"{stem}.npy",
            cam_dir / f"{image_id}_{stem}.npy",
        ]

        for path in exact_candidates:
            if path.exists():
                return np.load(path), path

        search_roots = []
        if raw_dir.exists():
            search_roots.append(raw_dir)
        search_roots.append(cam_dir)

        for root in search_roots:
            patterns = [
                f"{image_id}/{stem}.npy",
                f"{image_id}*{stem}*.npy",
                f"*{image_id}*{stem}*.npy",
            ]
            for pat in patterns:
                matches = sorted(root.glob(pat))
                if matches:
                    return np.load(matches[0]), matches[0]

    return None, None


def resize_mask_to_cam(mask: np.ndarray, cam_shape: tuple[int, int]) -> np.ndarray:
    return cv2.resize(mask.astype(np.uint8), (cam_shape[1], cam_shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)


def cue_localization_metrics_for_folder(cam_dir: Path, csv_path: Path, cam_name="finercam"):
    df = pd.read_csv(csv_path)
    df = df[(df["split"] == "test") & (df["gt_label"] == "MEL") & (df["cue_applied"] == True)].copy()

    rows = []
    for _, row in df.iterrows():
        image_id = str(row["image_id"])
        cam, cam_path = load_cam_array(cam_dir, image_id, candidates=(cam_name,))
        if cam is None:
            rows.append({"image_id": image_id, "status": "missing_cam"})
            continue

        cam = np.asarray(cam).squeeze().astype(float)
        cam = cam - np.nanmin(cam)
        denom = np.nanmax(cam) + 1e-8
        cam = cam / denom

        cue_mask_path = resolve_rel_path(HAM_ROOT, row["cue_mask_rel_path"])
        lesion_mask_path = resolve_rel_path(HAM_ROOT, row["mask_rel_path"])
        cue_mask = resize_mask_to_cam(load_mask_as_bool(cue_mask_path), cam.shape)
        lesion_mask = resize_mask_to_cam(load_mask_as_bool(lesion_mask_path), cam.shape)

        total = float(cam.sum()) + 1e-8
        cue_energy_fraction = float(cam[cue_mask].sum() / total) if cue_mask.any() else np.nan
        lesion_energy_fraction = float(cam[lesion_mask].sum() / total) if lesion_mask.any() else np.nan

        max_y, max_x = np.unravel_index(np.nanargmax(cam), cam.shape)
        cue_pointing_game = bool(cue_mask[max_y, max_x]) if cue_mask.any() else False
        lesion_pointing_game = bool(lesion_mask[max_y, max_x]) if lesion_mask.any() else False

        rows.append({
            "image_id": image_id,
            "cam_path": str(cam_path.relative_to(REPO_ROOT)) if cam_path else "",
            "status": "ok",
            "cue_energy_fraction": cue_energy_fraction,
            "lesion_energy_fraction": lesion_energy_fraction,
            "cue_pointing_game": cue_pointing_game,
            "lesion_pointing_game": lesion_pointing_game,
            "cue_area_frac_of_lesion": row.get("cue_area_frac_of_lesion", np.nan),
        })

    metrics_df = pd.DataFrame(rows)
    return metrics_df

## 4. Generate cued qualitative CAM panels

In [5]:
generate_qualitative_cams(
    experiments=EXPERIMENTS_CUE,
    csv_path=CUE_QUAL_CSV,
    img_dir=IMG_DIR,
    num_samples=10,
    dry_run=False,
)


Running cue CAM generation: Cue CE

python -m scripts.generate_finer_cam_panderm --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_fixed_qualitative_10_seed42.csv --image_col image_rel_path --img_dir /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --gt_col gt_label --checkpoint /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-cue.pth --checkpoint_model_type panderm --class_names MEL,NV --out_dir /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_cue_mel_nv_small_fixed_seed42_block-4/cam_cue_ce_on_cued_test --num_samples 10 --method finercam --compare_mode gt_pair --A MEL --B NV --topk_compare 1 --alpha 0.8 --panel_items rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam --mask_root /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --mask_col mask_rel_path --save_raw_cams --target_block_index -4
[info]

## 5. Build cued qualitative PDF

In [6]:
CONFIG_DIR = REPO_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

cue_pdf_config = [
    {"name": name, "folder": str(cfg["out_dir"].relative_to(REPO_ROOT))}
    for name, cfg in EXPERIMENTS_CUE.items()
]

CUE_JSON = CONFIG_DIR / f"qualitative_cue_mel_nv_2experiments_seed{QUAL_SEED}.json"
CUE_JSON.write_text(json.dumps(cue_pdf_config, indent=2))

print("Saved:", CUE_JSON)
print(json.dumps(cue_pdf_config, indent=2))

build_qualitative_pdf(
    csv_path=CUE_QUAL_CSV,
    experiments_json_path=CUE_JSON,
    out_pdf=QUAL_ROOT / f"qualitative_cue_mel_nv_2experiments_seed{QUAL_SEED}.pdf",
    num_samples=10,
    dry_run=False,
)

Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/configs/qualitative_cue_mel_nv_2experiments_seed42.json
[
  {
    "name": "Cue CE",
    "folder": "outputs/qual_cue_mel_nv_small_fixed_seed42_block-4/cam_cue_ce_on_cued_test"
  },
  {
    "name": "Cue HA",
    "folder": "outputs/qual_cue_mel_nv_small_fixed_seed42_block-4/cam_cue_ha_on_cued_test"
  }
]

python -m scripts.make_qualitative_comparison_pdf --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue_small/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_fixed_qualitative_10_seed42.csv --image_col image_rel_path --gt_col gt_label --out_pdf /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_cue_mel_nv_small_fixed_seed42_block-4/qualitative_cue_mel_nv_2experiments_seed42.pdf --experiments_json_path /Users/choekyelnyungmartsang/Developer/master-thesis/configs/qualitative_cue_mel_nv_2experiments_seed42.json --num_samples 10 --missing_policy placeholder
Saved PDF: /Users/choekyelnyun

## 6. Cue localization metrics

This uses `cue_mask_rel_path` and measures whether the saved CAM arrays put energy on the synthetic cue.

If your CAM generation script does not save `.npy` arrays with names containing `finercam`, this cell will report `missing_cam`. In that case, we need to patch `scripts/generate_finer_cam_panderm.py` to save raw CAM arrays.


In [7]:
all_metrics = []

for exp_name, cfg in EXPERIMENTS_CUE.items():
    metrics_df = cue_localization_metrics_for_folder(
        cam_dir=cfg["out_dir"],
        csv_path=CUE_QUAL_CSV,
        cam_name="finercam",
    )
    metrics_df.insert(0, "experiment", exp_name)
    all_metrics.append(metrics_df)

metrics_df = pd.concat(all_metrics, axis=0, ignore_index=True)
metrics_out = QUAL_ROOT / f"cue_localization_metrics_finercam_qual10_seed{QUAL_SEED}.csv"
metrics_df.to_csv(metrics_out, index=False)

print("Saved:", metrics_out)
display(metrics_df.head(20))
display(metrics_df["status"].value_counts())

ok_df = metrics_df[metrics_df["status"] == "ok"].copy()

if len(ok_df) == 0:
    print("No valid CAM arrays found.")
    print("Rerun CAM generation with --save_raw_cams.")
else:
    summary = ok_df.groupby("experiment").agg(
        n=("image_id", "count"),
        cue_energy_fraction_mean=("cue_energy_fraction", "mean"),
        cue_energy_fraction_median=("cue_energy_fraction", "median"),
        cue_pointing_game_rate=("cue_pointing_game", "mean"),
        lesion_energy_fraction_mean=("lesion_energy_fraction", "mean"),
        lesion_pointing_game_rate=("lesion_pointing_game", "mean"),
        cue_area_frac_mean=("cue_area_frac_of_lesion", "mean"),
    )
    display(summary)

Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_cue_mel_nv_small_fixed_seed42_block-4/cue_localization_metrics_finercam_qual10_seed42.csv


,experiment,image_id,cam_path,status,cue_energy_fraction,lesion_energy_fraction,cue_pointing_game,lesion_pointing_game,cue_area_frac_of_lesion
0,Cue CE,ISIC_0024459,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.321852,0.969094,False,True,0.042592
1,Cue CE,ISIC_0024756,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.296734,0.748547,True,True,0.105527
2,Cue CE,ISIC_0026993,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.312956,0.787475,True,True,0.103086
3,Cue CE,ISIC_0030798,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.317250,0.931987,False,True,0.043577
4,Cue CE,ISIC_0031408,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.298179,0.870054,True,True,0.039752
5,Cue HA,ISIC_0024459,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.294954,0.972784,False,True,0.042592
6,Cue HA,ISIC_0024756,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.158111,0.583957,True,True,0.105527
7,Cue HA,ISIC_0026993,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.223481,0.543632,True,True,0.103086
8,Cue HA,ISIC_0030798,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.248201,0.866890,False,True,0.043577
9,Cue HA,ISIC_0031408,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.168104,0.778893,True,True,0.039752


status
ok    10
Name: count, dtype: int64

,n,cue_energy_fraction_mean,cue_energy_fraction_median,cue_pointing_game_rate,lesion_energy_fraction_mean,lesion_pointing_game_rate,cue_area_frac_mean
experiment,,,,,,,
Cue CE,5,0.309394,0.312956,0.6,0.861432,1.0,0.066907
Cue HA,5,0.218570,0.223481,0.6,0.749231,1.0,0.066907


[small Cue CE]: The cue covers around **6.7** % of the lesion area, but FinerCAM assigns around **31** % of its heat to the cue. That is strong evidence that the model explanation is detecting the cue as discriminative.

[small Cue HA]: The cue covers around **6.7** % of the lesion area, but FinerCAM assigns around **22** % of its heat to the cue. This is also above chance, but lower than Cue CE. This suggests HA may reduce shortcut focus somewhat, or shift heat more broadly inside the lesion.

[big Cue CE]: The cue covers around **27** % of the lesion area, and FinerCAM assigns around **38** % of its heat to the cue.

[big Cue HA]: The cue covers around **27** % of the lesion area, and FinerCAM assigns around **35** % of its heat to the cue.

In [8]:
cam_names = ["gradcam_a", "gradcam_b", "map_diff", "finercam"]
all_method_metrics = []

for cam_name in cam_names:
    for exp_name, cfg in EXPERIMENTS_CUE.items():
        metrics_df = cue_localization_metrics_for_folder(
            cam_dir=cfg["out_dir"],
            csv_path=CUE_QUAL_CSV,
            cam_name=cam_name,
        )
        metrics_df.insert(0, "experiment", exp_name)
        metrics_df.insert(1, "cam_name", cam_name)
        all_method_metrics.append(metrics_df)

method_metrics_df = pd.concat(all_method_metrics, axis=0, ignore_index=True)
method_metrics_out = QUAL_ROOT / f"cue_localization_metrics_all_methods_seed{QUAL_SEED}.csv"
method_metrics_df.to_csv(method_metrics_out, index=False)

print("Saved:", method_metrics_out)

ok_methods = method_metrics_df[method_metrics_df["status"] == "ok"].copy()

if len(ok_methods) == 0:
    print("No valid CAM arrays found.")
    display(method_metrics_df["status"].value_counts())
else:
    method_summary = ok_methods.groupby(["experiment", "cam_name"]).agg(
        n=("image_id", "count"),
        cue_energy_fraction_mean=("cue_energy_fraction", "mean"),
        cue_energy_fraction_median=("cue_energy_fraction", "median"),
        cue_pointing_game_rate=("cue_pointing_game", "mean"),
        lesion_energy_fraction_mean=("lesion_energy_fraction", "mean"),
        lesion_pointing_game_rate=("lesion_pointing_game", "mean"),
        cue_area_frac_mean=("cue_area_frac_of_lesion", "mean"),
    ).reset_index()

    display(method_summary)

Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_cue_mel_nv_small_fixed_seed42_block-4/cue_localization_metrics_all_methods_seed42.csv


,experiment,cam_name,n,cue_energy_fraction_mean,cue_energy_fraction_median,cue_pointing_game_rate,lesion_energy_fraction_mean,lesion_pointing_game_rate,cue_area_frac_mean
0,Cue CE,finercam,5,0.309394,0.312956,0.6,0.861432,1.0,0.066907
1,Cue CE,gradcam_a,5,0.278936,0.288844,0.6,0.814893,1.0,0.066907
2,Cue CE,gradcam_b,5,0.000343,0.000460,0.0,0.310226,0.0,0.066907
3,Cue CE,map_diff,5,0.409641,0.403533,0.6,0.935643,1.0,0.066907
4,Cue HA,finercam,5,0.218570,0.223481,0.6,0.749231,1.0,0.066907
5,Cue HA,gradcam_a,5,0.089792,0.049812,0.0,0.440392,0.4,0.066907
6,Cue HA,gradcam_b,5,0.000006,0.000004,0.0,0.260548,0.0,0.066907
7,Cue HA,map_diff,5,0.122179,0.087180,0.0,0.524035,0.6,0.066907


## Interpretation checklist

Expected if the model strongly relies on the cue and FinerCAM reveals it:

- `cue_energy_fraction` should be clearly above the physical cue area fraction.
- `cue_pointing_game_rate` should be high.
- Cue CE should be at least as cue-focused as Cue HA.

If classification is perfect but cue localization is low, then the model may use the cue in a way that final-layer CAMs do not localize clearly, or the cue is too small relative to the final feature resolution.


In [ ]:
# ============================================================
# Layer sensitivity test for CAM localization
# ============================================================

LAYER_TEST_ROOT = QUAL_ROOT / f"layer_sensitivity_seed{QUAL_SEED}"
LAYER_TEST_ROOT.mkdir(parents=True, exist_ok=True)

# Test only Cue CE first.
# If this is informative, repeat for Cue HA later.
LAYER_SENSITIVITY_EXPERIMENTS = {
    "Cue CE": EXPERIMENTS_CUE["Cue CE"],
}

# ViT base has 12 blocks, so:
# -1 resolves to block 11
# -4 resolves to block 8
# -6 resolves to block 6
# -8 resolves to block 4
# TARGET_BLOCKS = [-1, -4, -6, -8]
TARGET_BLOCKS = [-1, -2, -3, -4, -5, -6, -7, -8]

def generate_layer_sensitivity_cams(
    experiments: dict,
    csv_path: Path,
    img_dir: Path,
    target_blocks: list[int],
    num_samples: int = 10,
    dry_run: bool = False,
):
    for block_idx in target_blocks:
        for exp_name, cfg in experiments.items():
            safe_exp_name = exp_name.lower().replace(" ", "_")
            out_dir = LAYER_TEST_ROOT / f"{safe_exp_name}_block{block_idx}"
            out_dir.mkdir(parents=True, exist_ok=True)

            panel_items = "rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam"

            cmd = [
                "python", "-m", "scripts.generate_finer_cam_panderm",
                "--csv", str(csv_path),
                "--image_col", "image_rel_path",
                "--img_dir", str(img_dir),
                "--gt_col", "gt_label",
                "--checkpoint", str(cfg["checkpoint"]),
                "--checkpoint_model_type", cfg["checkpoint_model_type"],
                "--class_names", "MEL,NV",
                "--out_dir", str(out_dir),
                "--num_samples", str(num_samples),
                "--method", "finercam",
                "--compare_mode", "gt_pair",
                "--A", "MEL",
                "--B", "NV",
                "--topk_compare", "1",
                "--alpha", "0.8",
                "--panel_items", panel_items,
                "--mask_root", str(MASK_ROOT),
                "--mask_col", "mask_rel_path",
                "--save_json",
                "--save_raw_cams",
                "--target_block_index", str(block_idx),
            ]

            print(f"\nRunning layer sensitivity: {exp_name}, target_block_index={block_idx}")
            run_command(cmd, dry_run=dry_run)


generate_layer_sensitivity_cams(
    experiments=LAYER_SENSITIVITY_EXPERIMENTS,
    csv_path=CUE_QUAL_CSV,
    img_dir=IMG_DIR,
    target_blocks=TARGET_BLOCKS,
    num_samples=10,
    dry_run=False,
)

In [10]:
# ============================================================
# Cue localization metrics across target CAM layers
# ============================================================

layer_metrics = []

for block_idx in TARGET_BLOCKS:
    for exp_name in LAYER_SENSITIVITY_EXPERIMENTS.keys():
        safe_exp_name = exp_name.lower().replace(" ", "_")
        cam_dir = LAYER_TEST_ROOT / f"{safe_exp_name}_block{block_idx}"

        for cam_name in ["gradcam_a", "map_diff", "finercam"]:
            metrics_df = cue_localization_metrics_for_folder(
                cam_dir=cam_dir,
                csv_path=CUE_QUAL_CSV,
                cam_name=cam_name,
            )

            metrics_df.insert(0, "experiment", exp_name)
            metrics_df.insert(1, "target_block_index", block_idx)
            metrics_df.insert(2, "cam_name", cam_name)
            layer_metrics.append(metrics_df)

layer_metrics_df = pd.concat(layer_metrics, axis=0, ignore_index=True)

layer_metrics_out = LAYER_TEST_ROOT / f"layer_sensitivity_cue_metrics_seed{QUAL_SEED}.csv"
layer_metrics_df.to_csv(layer_metrics_out, index=False)

print("Saved:", layer_metrics_out)
pd.set_option("display.float_format", "{:.4f}".format)
display(layer_metrics_df.head(20))
display(layer_metrics_df["status"].value_counts())

ok_layer_df = layer_metrics_df[layer_metrics_df["status"] == "ok"].copy()

if len(ok_layer_df) == 0:
    print("No valid CAM arrays found.")
else:
    layer_summary = ok_layer_df.groupby(
        ["experiment", "target_block_index", "cam_name"]
    ).agg(
        n=("image_id", "count"),
        cue_energy_fraction_mean=("cue_energy_fraction", "mean"),
        cue_energy_fraction_median=("cue_energy_fraction", "median"),
        cue_pointing_game_rate=("cue_pointing_game", "mean"),
        lesion_energy_fraction_mean=("lesion_energy_fraction", "mean"),
        lesion_pointing_game_rate=("lesion_pointing_game", "mean"),
        cue_area_frac_mean=("cue_area_frac_of_lesion", "mean"),
    ).reset_index()

    display(layer_summary.sort_values(
        ["cam_name", "cue_energy_fraction_mean"],
        ascending=[True, False],
    ))

Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/qual_cue_mel_nv_small_fixed_seed42_block-4/layer_sensitivity_seed42/layer_sensitivity_cue_metrics_seed42.csv


,experiment,target_block_index,cam_name,image_id,cam_path,status,cue_energy_fraction,lesion_energy_fraction,cue_pointing_game,lesion_pointing_game,cue_area_frac_of_lesion
0,Cue CE,-1,gradcam_a,ISIC_0024459,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.0048,0.4741,False,False,0.0426
1,Cue CE,-1,gradcam_a,ISIC_0024756,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.0050,0.1779,False,False,0.1055
2,Cue CE,-1,gradcam_a,ISIC_0026993,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.0034,0.1968,False,False,0.1031
3,Cue CE,-1,gradcam_a,ISIC_0030798,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.0028,0.5096,False,True,0.0436
4,Cue CE,-1,gradcam_a,ISIC_0031408,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.0027,0.5545,False,False,0.0398
5,Cue CE,-1,map_diff,ISIC_0024459,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.0048,0.4741,False,False,0.0426
6,Cue CE,-1,map_diff,ISIC_0024756,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.0050,0.1779,False,False,0.1055
7,Cue CE,-1,map_diff,ISIC_0026993,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.0034,0.1968,False,False,0.1031
8,Cue CE,-1,map_diff,ISIC_0030798,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.0028,0.5096,False,True,0.0436
9,Cue CE,-1,map_diff,ISIC_0031408,outputs/qual_cue_mel_nv_small_fixed_seed42_blo...,ok,0.0027,0.5545,False,False,0.0398


status
ok    120
Name: count, dtype: int64

,experiment,target_block_index,cam_name,n,cue_energy_fraction_mean,cue_energy_fraction_median,cue_pointing_game_rate,lesion_energy_fraction_mean,lesion_pointing_game_rate,cue_area_frac_mean
12,Cue CE,-4,finercam,5,0.3094,0.3130,0.6000,0.8614,1.0000,0.0669
15,Cue CE,-3,finercam,5,0.2553,0.2458,0.8000,0.8293,1.0000,0.0669
9,Cue CE,-5,finercam,5,0.0678,0.0621,0.0000,0.5457,0.6000,0.0669
0,Cue CE,-8,finercam,5,0.0060,0.0016,0.0000,0.4739,0.4000,0.0669
21,Cue CE,-1,finercam,5,0.0041,0.0037,0.0000,0.3977,0.4000,0.0669
18,Cue CE,-2,finercam,5,0.0029,0.0025,0.0000,0.5153,0.4000,0.0669
6,Cue CE,-6,finercam,5,0.0014,0.0001,0.0000,0.6013,1.0000,0.0669
3,Cue CE,-7,finercam,5,0.0007,0.0002,0.0000,0.3909,0.2000,0.0669
13,Cue CE,-4,gradcam_a,5,0.2789,0.2888,0.6000,0.8149,1.0000,0.0669
16,Cue CE,-3,gradcam_a,5,0.1529,0.1750,0.0000,0.7829,1.0000,0.0669


CAM layer sensitivity check! -4 or -2 works well